In [21]:
import polars as pl

## Generate the Sample Dataset

Let's create a sample dataset with a categorical feature `Color` and a numerical feature `Value`.

In [22]:
data = {
    'Color': ['Red', 'Green', 'Blue', 'Purple', 'Green'],
    'Value': [10, 20, 15, 25, 30],
}

df = pl.DataFrame(data)

## One-Hot Encoding

We will use `to_dummies` to one-hot encode the `Color` column.

In [23]:
df_encoded = df.to_dummies(columns=['Color'])

In [24]:
df_encoded

Color_Blue,Color_Green,Color_Purple,Color_Red,Value
u8,u8,u8,u8,i64
0,0,0,1,10
0,1,0,0,20
1,0,0,0,15
0,0,1,0,25
0,1,0,0,30


## Create the Holdout Dataset

Now, let's create a holdout dataset that might not have all the same categories in the `Color` feature.

In [25]:
holdout_data = {
    'Color': ['Red', 'Blue', 'Yellow', 'Green'],
    'Value': [18, 22, 28, 35],
}

holdout_df = pl.DataFrame(holdout_data)

In [26]:
holdout_encoded = holdout_df.to_dummies(columns=['Color'])

The one-hot-encoded holdout dataset looks like this:

In [27]:
holdout_encoded

Color_Blue,Color_Green,Color_Red,Color_Yellow,Value
u8,u8,u8,u8,i64
0,0,1,0,18
1,0,0,0,22
0,0,0,1,28
0,1,0,0,35


Notice that the original training dataset contains `Color_Purple`.

In [28]:
df_encoded.schema

Schema([('Color_Blue', UInt8),
        ('Color_Green', UInt8),
        ('Color_Purple', UInt8),
        ('Color_Red', UInt8),
        ('Value', Int64)])

Notice that this holdout DataFrame does not contain `Color_Purple`, but it does contain `Color_Yellow`.

In [29]:
holdout_encoded.schema

Schema([('Color_Blue', UInt8),
        ('Color_Green', UInt8),
        ('Color_Red', UInt8),
        ('Color_Yellow', UInt8),
        ('Value', Int64)])

## Align the Holdout Dataset

In Polars, a standard eager pattern is to add any columns that appear in the training data but are missing from the holdout data, and then select the training columns so the order matches and any extra holdout-only categories are dropped.

Here that means we will add `Color_Purple` with zeros and remove `Color_Yellow`.

In [30]:
def align_to_training_columns(train_encoded: pl.DataFrame, other_encoded: pl.DataFrame) -> pl.DataFrame:
    missing_columns = [
        pl.lit(0).cast(train_encoded.schema[column]).alias(column)
        for column in train_encoded.columns
        if column not in other_encoded.columns
    ]

    return (
        other_encoded
        .with_columns(missing_columns)
        .select(train_encoded.columns)
    )

holdout_aligned = align_to_training_columns(df_encoded, holdout_encoded)

In [31]:
df_encoded

Color_Blue,Color_Green,Color_Purple,Color_Red,Value
u8,u8,u8,u8,i64
0,0,0,1,10
0,1,0,0,20
1,0,0,0,15
0,0,1,0,25
0,1,0,0,30


In [32]:
holdout_aligned

Color_Blue,Color_Green,Color_Purple,Color_Red,Value
u8,u8,u8,u8,i64
0,0,0,1,18
1,0,0,0,22
0,0,0,0,28
0,1,0,0,35


## Alternate Version

Polars has another function that is currently *unstable*, meaning it could be removed from future versions. The function is called `match_to_schema()` and allows adding, removing, and resorting columns in a single call.


In [33]:
holdout_matched = holdout_encoded.match_to_schema(df_encoded.schema, missing_columns=pl.lit(0), extra_columns='ignore')

In [34]:
holdout_matched

Color_Blue,Color_Green,Color_Purple,Color_Red,Value
u8,u8,i32,u8,i64
0,0,0,1,18
1,0,0,0,22
0,0,0,0,28
0,1,0,0,35


## Complete

Now that both DataFrames have the same columns we can make predictions on the holdout dataset.